# Part 8 · Notebook 10 — Position sizing and the Kelly criterion

**Sessions:** S19 (Position sizing I: fixed risk & volatility) · S20 (Position sizing II: Kelly & estimation error) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Size a trade so that its stop loses a fixed fraction of equity.
2. Simulate the risk of ruin for different bet sizes.
3. Compute the Kelly fraction, and see why full Kelly with an estimated edge is dangerous.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. Fixed-fractional sizing

Decide how much of equity one trade may lose (say 1%), and let the stop distance set the size: `units = floor(equity × risk_frac / (|entry − stop| × multiplier))`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def risk_per_trade_size(equity, risk_frac, entry, stop, multiplier=1.0):
    return ...                                    # ✍️ an int (floored)

cases = [(100_000, 0.01, 50.0, 48.0), (100_000, 0.01, 50.0, 45.0), (250_000, 0.005, 5000.0, 4980.0, 50.0), (250_000, 0.005, 5000.0, 4980.0, 5.0)]
mine = [p.attempt(risk_per_trade_size, *cs) for cs in cases]
mine = p.check("risk_per_trade_size", mine, [p.risk_per_trade_size(*cs) for cs in cases])
pd.DataFrame({"case": ["stock, stop 2 below", "stock, stop 5 below", "ES, stop 20 points", "MES, stop 20 points"], "units": mine})

A wider stop means a smaller position, so every trade risks the same. With ES a $1,250 risk budget can't even buy one contract; micros can size it properly.

## 2. Risk of ruin

A system that wins 40% of the time, winning 2R and losing 1R: a positive expectancy of 0.2R per trade. Simulate `n_sims` sequences of `n_trades` results (R-multiples drawn with replacement), equity `×= 1 + risk_frac·R`, and report the share of paths whose equity **ever** falls to `ruin` (0.5 = a 50% drawdown from the start).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def risk_of_ruin(r_multiples, risk_frac, ruin=0.5, n_trades=500, n_sims=2000, seed=0):
    rng = np.random.default_rng(seed)
    R = rng.choice(np.asarray(r_multiples, dtype=float), size=(n_sims, n_trades))
    eq = ...                                      # ✍️ the equity path of every simulation (cumulative product)
    return float(np.mean(eq.min(axis=1) <= ruin))

R = np.array([-1.0] * 60 + [2.0] * 40)
fracs = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
mine = [p.attempt(risk_of_ruin, R, f) for f in fracs]
mine = p.check("risk_of_ruin", mine, [p.risk_of_ruin(R, f) for f in fracs])
pd.Series(mine, index=[f"risk {f:.1%} per trade" for f in fracs], name="P(ever down 50%)").round(3)

Same edge, same trades: at 2% per trade the account is essentially safe, at 10% it halves in almost every other path.

## 3. Kelly

The **Kelly** fraction maximizes long-run growth:
* discrete bets (win probability `p`, win/loss ratio `b`): `f* = p − (1 − p)/b`;
* continuous returns (mean `μ`, volatility `σ`, rate `r`): leverage `(μ − r)/σ²`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def kelly_discrete(p_win, win_loss_ratio):
    return ...                                    # ✍️

def kelly_continuous(mu, sigma, r=0.0):
    return ...                                    # ✍️

mine = [kelly_discrete(0.4, 2.0), kelly_discrete(0.55, 1.0), kelly_continuous(0.08, 0.16), kelly_continuous(0.08, 0.16, 0.03)]
mine = p.check("kelly", mine, [p.kelly_discrete(0.4, 2.0), p.kelly_discrete(0.55, 1.0), p.kelly_continuous(0.08, 0.16), p.kelly_continuous(0.08, 0.16, 0.03)])
dict(zip(["40% × 2R", "55% even money", "equity 8%/16%", "equity, 3% rate"], np.round(mine, 3)))

The 40%/2R system's Kelly bet is exactly the 10% that gave a 46% chance of halving the account above: Kelly maximizes growth, not comfort. And an equity index with an 8% edge and 16% vol has a Kelly leverage above 3.

Worse, `μ` is **estimated**. Estimate it from five years of data, lever at a fraction of the estimated Kelly, and run twenty years of the true process:

In [ ]:
sim = p.kelly_growth_simulation(n_paths=500)
sim.index = [f"{f:g}× Kelly" for f in sim.index]
sim.round(3)

Half Kelly grows the median investor the most here, and with far smaller drawdowns; full Kelly on an estimated edge loses money for about a third of investors; double Kelly ruins most of them. Use fractional Kelly as a **cap**, and volatility targeting for the day-to-day size.

## Wrap-up

* Size by risk (stop distance, ATR, volatility), not by conviction.
* Simulate ruin before choosing the fraction.
* Kelly with an estimated edge overbets; half Kelly or less.
* Graded version: `labs/part08/week29_risk_sizing` (volatility targeting, Turtle units, multi-asset Kelly).